# Fase 3 · M04: Encoding — Índice

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M04 — Encoding (Índice) |

---

## 🎯 Qué hace

Índice de los dos notebooks de encoding: M04a (100% numérico para AutoML) y M04b (mixto para EDA). Verifica los datasets generados.

## 📋 Requisitos

- `data/03_features/df_exp_automl_target.parquet`
- `data/03_features/df_exp_target_eda.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase3/m04_encoding.html` | Índice de encoding |

## 🔄 Flujo

```
df_exp_automl_target.parquet + df_exp_target_eda.parquet
    ↓ Verificación de shapes y tipos
    → docs/html/fase3/m04_encoding.html
```

## ➡️ Siguiente

`f3_m05_target_export.ipynb` — definición del target y exportación final


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y CARGAR NÚMEROS REALES (SIN HARDCODING)
# ============================================================================
# Lee los datasets generados por M04a y M04b para mostrar
# números reales en el HTML — sin valores hardcodeados.
# Si los archivos no existen aún, muestra aviso y continúa.
# ============================================================================

import sys
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_FEATURES, info_entorno

# M04a — 100% numérico
try:
    df_automl = pd.read_parquet(RUTA_FEATURES / 'df_exp_automl_target.parquet')
    cols_automl = len(df_automl.columns)
    strings_automl = df_automl.select_dtypes(include='object').shape[1]
    print(f'✓ M04a: {cols_automl} columnas, {strings_automl} strings (debe ser 0)')
except Exception as e:
    cols_automl = None
    strings_automl = 0
    print(f'⚠️  M04a no encontrado: {e}')

# M04b — mixto
try:
    df_eda = pd.read_parquet(RUTA_FEATURES / 'df_exp_target_eda.parquet')
    cols_eda = len(df_eda.columns)
    strings_eda = df_eda.select_dtypes(include='object').shape[1]
    nums_eda = df_eda.select_dtypes(include='number').shape[1]
    print(f'✓ M04b: {cols_eda} columnas ({strings_eda} strings + {nums_eda} numéricos)')
except Exception as e:
    cols_eda = None
    strings_eda = 0
    nums_eda = 0
    print(f'⚠️  M04b no encontrado: {e}')

info_entorno()
print('\n✅ Números cargados dinámicamente')


✓ M04a: 49 columnas, 1 strings (debe ser 0)
✓ M04b: 49 columnas (10 strings + 39 numéricos)
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\FF\AU_UJI_v2
✓ 📁 RAW:           C:\FF\AU_UJI_v2\data\00_raw
✓ 📁 INTERIM:       C:\FF\AU_UJI_v2\data\01_interim
✓ 📁 PROCESSED:     C:\FF\AU_UJI_v2\data\02_processed
✓ 📁 FEATURES:      C:\FF\AU_UJI_v2\data\03_features
✓ 📁 AUTOML:        C:\FF\AU_UJI_v2\data\automl
✓ 📁 NOTEBOOKS:     C:\FF\AU_UJI_v2\notebooks
✓ 📄 Excel principal: C:\FF\AU_UJI_v2\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ===========================================================================

✅ Números cargados dinámicamente


In [2]:
# ============================================================================
# CELDA 2: GENERAR HTML ÍNDICE M04
# ============================================================================

from src.config import RUTA_HTML
from src.html import (
    generar_kpis_html, generar_tarjetas_html,
    generar_seccion_html, generar_html_navegacion_completa, guardar_html
)
from src.html.render import render_pagina_desde_fichero
from src.utils import crear_directorios

RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FASE3_HTML])

print('='*70)
print('F3-M04: ÍNDICE — ENCODING DE VARIABLES CATEGÓRICAS')
print('='*70)

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase3', modulo_activo='m04'
)

# KPIs dinámicos
KPIS = [
    {'valor': str(cols_automl) if cols_automl else '—', 'titulo': 'Cols AutoML'},
    {'valor': str(cols_eda) if cols_eda else '—', 'titulo': 'Cols EDA'},
    {'valor': '0', 'titulo': 'Strings en AutoML'},
    {'valor': str(strings_eda) if cols_eda else '—', 'titulo': 'Strings en EDA'},
]
kpis_html = generar_kpis_html(KPIS)

# Tarjetas
desc_automl = f'{cols_automl} columnas 100% numéricas.' if cols_automl else 'Ejecutar M04a primero.'
desc_eda = f'{strings_eda} strings + {nums_eda} numéricos = {cols_eda} cols.' if cols_eda else 'Ejecutar M04b primero.'

tarjetas = [
    {
        'titulo': 'M04a — 100% Numérico (AutoML)',
        'descripcion': f'{desc_automl} Mapas de encoding desde src/config_datos.py. Listo para Stacking, PyCaret, H2O, AutoGluon.',
        'emoji': '🤖',
        'link': 'm04_automl.html',
        'link_texto': 'Ver detalles →',
        'color': '#3182ce'
    },
    {
        'titulo': 'M04b — Mixto (EDA + CatBoost)',
        'descripcion': f'{desc_eda} Strings nativos para CatBoost. titulacion y cupo legibles para EDA.',
        'emoji': '📊',
        'link': 'm04_eda.html',
        'link_texto': 'Ver detalles →',
        'color': '#ed8936'
    }
]
tarjetas_html = generar_tarjetas_html(tarjetas)
s_datasets = generar_seccion_html('📦 Dos versiones de encoding', tarjetas_html, '🔧')

# Comparativa
s_comparativa = generar_seccion_html('📋 Comparativa', '''
<table style="width:100%;border-collapse:collapse;">
<tr style="background:#3182ce;color:white;">
  <th style="padding:10px;text-align:left;">Aspecto</th>
  <th style="text-align:center;">M04a — AutoML</th>
  <th style="text-align:center;">M04b — EDA/CatBoost</th>
</tr>
<tr><td style="padding:8px;">Strings</td><td style="text-align:center;">0 — todo numérico</td><td style="text-align:center;">Mantiene strings legibles</td></tr>
<tr style="background:#f7fafc;"><td style="padding:8px;">titulacion</td><td style="text-align:center;">Eliminada (M05 añade tasa_abandono)</td><td style="text-align:center;">String nativo para CatBoost</td></tr>
<tr><td style="padding:8px;">cupo</td><td style="text-align:center;">Codificado (0-7)</td><td style="text-align:center;">String legible</td></tr>
<tr style="background:#f7fafc;"><td style="padding:8px;">Encoding</td><td style="text-align:center;">Mapas fijos desde config_datos.py</td><td style="text-align:center;">Solo booleans y floats</td></tr>
<tr><td style="padding:8px;">Uso</td><td style="text-align:center;">Stacking, AutoML, RF, LogReg</td><td style="text-align:center;">CatBoost, EDA, gráficos</td></tr>
<tr style="background:#f7fafc;"><td style="padding:8px;">Siguiente</td><td style="text-align:center;">M05 → D_strict</td><td style="text-align:center;">M05 → df_eda_con_target</td></tr>
</table>''')

# Nota sobre mapas de encoding
s_nota = generar_seccion_html('⚙️ Fuente única de encoding', '''
<div style="background:#f0f4f8;padding:15px;border-radius:10px;border-left:4px solid #3182ce;">
  <p><strong>Sin hardcodes.</strong> Todos los mapas de encoding (via_acceso, rama, sexo, provincia,
  pais_nombre, universidad_origen, situacion_laboral, cupo, egresado) están definidos
  en <code>src/config_datos.py</code>.</p>
  <p>Si un valor cambia de nombre entre años académicos (ej: renombrar una vía de acceso),
  solo hay que añadir la variante en <code>config_datos.py</code> — funciona
  automáticamente en M04a y en la app Streamlit.</p>
</div>''')

html = render_pagina_desde_fichero(
    'f3_m04_index.ipynb',
    kpis_html + s_datasets + s_comparativa + s_nota,
    carpeta_notebook='fase3_features'
)

ruta_html = RUTA_FASE3_HTML / 'm04_encoding.html'
guardar_html(html, ruta_html)
print(f'✅ INDEX generado: {ruta_html}')


✓ Directorios verificados: 1
F3-M04: ÍNDICE — ENCODING DE VARIABLES CATEGÓRICAS
✅ HTML guardado: C:\FF\AU_UJI_v2\docs\html\fase3\m04_encoding.html
✅ INDEX generado: C:\FF\AU_UJI_v2\docs\html\fase3\m04_encoding.html
